# Video Pipeline: VSS + ASR

This notebook runs two independent analysis steps over the same video clip:

1. VSS video understanding for visual context and answering the user question.
2. Riva ASR transcription for the audio track using `grpcurl` and PCM audio.

The notebook ends with two separate outputs that can be passed into a third LLM call later.

# Setup and Imports

In [16]:
import base64
import json
import re
import shutil
import subprocess
from pathlib import Path

import requests
from IPython.display import Markdown, display

VSS_BASE_URL = "http://127.0.0.1:8000"
CHAT_URL = f"{VSS_BASE_URL}/v1/chat"
VIDEO_API_URL = f"{VSS_BASE_URL}/api/v1/videos"
RIVA_ENDPOINT = "localhost:51051"
NEMOTRON_BASE_URL = "http://127.0.0.1:30081/v1"
NEMOTRON_DEFAULT_MODEL = "nvidia/nvidia-nemotron-nano-9b-v2"
REQUEST_TIMEOUT = 300
PCM_SAMPLE_RATE = 16000
PCM_CHANNELS = 1
PCM_OUTPUT_PATH = Path("/tmp/test-200s.pcm")


# User Inputs

In [17]:
# Video input used by both VSS and ASR
video_path = "/home/ubuntu/sample-data/videoplayback.mp4"

# Question the user wants to search/ask about the video
user_prompt = "Give me an extensive and detailed summary of the video."

# Shared time window for both stages
# Use "HH:MM:SS" or "HH:MM:SS.sss".
start_time = "00:00:08"
end_time = None


## Nemotron Prompt Configuration

In [18]:
NEMOTRON_SYSTEM_PROMPT = (
    "You synthesize VSS and ASR outputs into one final answer. "
    "Return only the final answer. Do not reveal reasoning, chain-of-thought, or intermediate analysis."
)


def format_asr_segments(asr_segments):
    if not asr_segments:
        return ""

    lines = []
    for segment in asr_segments:
        speaker = segment.get("speaker")
        speaker_label = f"Speaker {speaker}" if speaker is not None else "Speaker ?"
        lines.append(
            f"[{segment.get('start_time')} - {segment.get('end_time')}] "
            f"{speaker_label}: {segment.get('text', '')}"
        )

    return "\n".join(lines)


def build_nemotron_messages(question, vss_answer, asr_transcript, asr_segments=None):
    diarized_context = format_asr_segments(asr_segments)

    user_prompt = f"""Use the following context to answer the question.

Question:
{question}

VSS answer:
{vss_answer}

ASR transcript:
{asr_transcript}

ASR diarization:
{diarized_context}

Return a direct answer only.
"""

    return [
        {
            "role": "system",
            "content": NEMOTRON_SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": user_prompt,
        },
    ]


# Validate Inputs and Helpers

In [19]:
def time_to_seconds(value):
    if value is None:
        return None

    if isinstance(value, (int, float)):
        return float(value)

    value = str(value).strip()
    parts = value.split(':')

    if len(parts) == 3:
        hours, minutes, seconds = parts
        return int(hours) * 3600 + int(minutes) * 60 + float(seconds)

    if len(parts) == 2:
        minutes, seconds = parts
        return int(minutes) * 60 + float(seconds)

    return float(value)


def validate_video_file(video_path):
    video_file = Path(video_path).expanduser().resolve()

    if not video_file.exists():
        raise FileNotFoundError(f"Video does not exist: {video_file}")

    if not video_file.is_file():
        raise ValueError(f"Path is not a file: {video_file}")

    if video_file.suffix.lower() != '.mp4':
        raise ValueError(f"Expected an MP4 file, got: {video_file.suffix}")

    return video_file


def build_video_understanding_prompt(sensor_id, user_prompt, start_time=None, end_time=None):
    parts = ["/video_understanding", "sensor_id", str(sensor_id)]

    if start_time is not None:
        parts.extend(["start_time", str(start_time)])

    if end_time is not None:
        parts.extend(["end_time", str(end_time)])

    parts.append(user_prompt)
    return " ".join(parts)


def extract_final_answer(content):
    if not content:
        return ""

    cleaned = re.sub(
        r"<agent-think>.*?</agent-think>",
        "",
        content,
        flags=re.DOTALL | re.IGNORECASE,
    )
    return cleaned.strip()


def extract_asr_transcript(riva_result):
    transcripts = []

    results = riva_result.get("results", []) if isinstance(riva_result, dict) else []
    for result in results:
        if not isinstance(result, dict):
            continue

        alternatives = result.get("alternatives", [])
        for alternative in alternatives:
            if isinstance(alternative, dict):
                transcript = alternative.get("transcript")
                if transcript:
                    transcripts.append(transcript)

        transcript = result.get("transcript")
        if transcript:
            transcripts.append(transcript)

    if not transcripts and isinstance(riva_result, dict):
        transcript = riva_result.get("transcript") or riva_result.get("text")
        if transcript:
            transcripts.append(transcript)

    return "\n".join(t.strip() for t in transcripts if t and t.strip())


video_file = validate_video_file(video_path)
start_seconds = time_to_seconds(start_time)
end_seconds = time_to_seconds(end_time)

if start_seconds is not None and start_seconds < 0:
    raise ValueError("start_time cannot be negative")

if end_seconds is not None and end_seconds < 0:
    raise ValueError("end_time cannot be negative")

if start_seconds is not None and end_seconds is not None and end_seconds <= start_seconds:
    raise ValueError("end_time must be greater than start_time")

print(f"Video: {video_file}")
print(f"Size: {video_file.stat().st_size:,} bytes")
print("start_seconds:", start_seconds)
print("end_seconds:", end_seconds)


Video: /home/ubuntu/sample-data/videoplayback.mp4
Size: 33,730,466 bytes
start_seconds: 8.0
end_seconds: None


## ASR Helpers

In [20]:
def ensure_tool_exists(tool_name):
    if shutil.which(tool_name) is None:
        raise RuntimeError(
            f"Required tool is not available on PATH: {tool_name}"
        )


def extract_pcm_from_video(
    video_file,
    pcm_file,
    start_time=None,
    end_time=None,
):
    ensure_tool_exists("ffmpeg")

    command = ["ffmpeg", "-y"]

    if start_time is not None:
        command.extend(["-ss", str(start_time)])

    command.extend([
        "-i", str(video_file),
        "-vn",
        "-ac", str(PCM_CHANNELS),
        "-ar", str(PCM_SAMPLE_RATE),
        "-f", "s16le",
    ])

    if start_time is not None and end_time is not None:
        start_seconds_local = time_to_seconds(start_time)
        end_seconds_local = time_to_seconds(end_time)

        if start_seconds_local is None or end_seconds_local is None:
            raise ValueError("start_time and end_time must be valid timestamps")

        duration_seconds = end_seconds_local - start_seconds_local

        if duration_seconds <= 0:
            raise ValueError("end_time must be greater than start_time")

        command.extend(["-t", str(duration_seconds)])

    command.append(str(pcm_file))

    subprocess.run(
        command,
        check=True,
        capture_output=True,
        text=True,
    )

    return pcm_file


def extract_diarized_segments(riva_result):
    segments = []

    for result in riva_result.get("results", []):
        alternatives = result.get("alternatives", [])
        if not alternatives:
            continue

        alternative = alternatives[0]
        words = alternative.get("words", [])
        transcript = alternative.get("transcript", "").strip()

        if not transcript:
            continue

        segments.append({
            "start_time": words[0].get("start_time") if words else None,
            "end_time": words[-1].get("end_time") if words else None,
            "speaker": next(
                (word.get("speaker_tag") for word in words if word.get("speaker_tag") not in (None, 0)),
                None,
            ),
            "text": transcript,
        })

    return segments


def extract_asr_transcript(riva_result):
    segments = extract_diarized_segments(riva_result)
    return "\n".join(segment["text"] for segment in segments if segment.get("text"))


def run_riva_asr(pcm_file):
    ensure_tool_exists("grpcurl")

    request = {
        "config": {
            "encoding": "LINEAR_PCM",
            "sample_rate_hertz": PCM_SAMPLE_RATE,
            "audio_channel_count": PCM_CHANNELS,
            "language_code": "en-US",
            "max_alternatives": 1,
            "enable_word_time_offsets": True,
            "enable_automatic_punctuation": True,
            "diarization_config": {
                "enable_speaker_diarization": True,
                "max_speaker_count": 4,
            },
        },
        "audio": base64.b64encode(Path(pcm_file).read_bytes()).decode("ascii"),
    }

    result = subprocess.run(
        [
            "grpcurl",
            "-plaintext",
            "-d",
            "@",
            RIVA_ENDPOINT,
            "nvidia.riva.asr.RivaSpeechRecognition/Recognize",
        ],
        input=json.dumps(request),
        text=True,
        capture_output=True,
        check=True,
    )

    riva_result = json.loads(result.stdout)
    return {
        "request": request,
        "raw_response": riva_result,
        "segments": extract_diarized_segments(riva_result),
        "transcript": extract_asr_transcript(riva_result),
    }


# VSS Pipeline

In [21]:
def run_vss_pipeline(video_file, user_prompt, start_time=None, end_time=None):
    response = requests.post(
        VIDEO_API_URL,
        json={"filename": str(video_file)},
        timeout=REQUEST_TIMEOUT,
    )
    response.raise_for_status()
    upload_info = response.json()

    upload_url = upload_info.get("url")
    if not upload_url:
        raise RuntimeError(f"VSS did not return an upload URL: {upload_info}")

    with video_file.open("rb") as file_handle:
        upload_response = requests.post(
            upload_url,
            files={"file": (video_file.name, file_handle, "video/mp4")},
            timeout=REQUEST_TIMEOUT,
        )
    upload_response.raise_for_status()

    vst_result = upload_response.json()
    sensor_id = vst_result.get("sensorId") or vst_result.get("sensor_id")
    if not sensor_id:
        raise RuntimeError(f"VST response did not contain sensorId: {vst_result}")

    filename = vst_result.get("filename") or video_file.stem
    complete_url = f"{VSS_BASE_URL}/api/v1/videos/{sensor_id}/complete"
    complete_response = requests.post(
        complete_url,
        json={"filename": filename},
        timeout=REQUEST_TIMEOUT,
    )
    complete_response.raise_for_status()
    complete_result = complete_response.json()

    prompt = build_video_understanding_prompt(
        sensor_id=sensor_id,
        user_prompt=user_prompt,
        start_time=start_time,
        end_time=end_time,
    )

    chat_response = requests.post(
        CHAT_URL,
        headers={"Content-Type": "application/json"},
        json={"messages": [{"role": "user", "content": prompt}]},
        timeout=REQUEST_TIMEOUT,
    )
    chat_response.raise_for_status()

    chat_result = chat_response.json()
    choices = chat_result.get("choices", [])
    if not choices:
        raise RuntimeError(f"/v1/chat returned no choices: {chat_result}")

    message = choices[0].get("message", {})
    raw_content = message.get("content", "")
    if not raw_content:
        raise RuntimeError(f"/v1/chat returned an empty message: {chat_result}")

    final_answer = extract_final_answer(raw_content)

    return {
        "sensor_id": sensor_id,
        "filename": filename,
        "prompt": prompt,
        "upload_info": upload_info,
        "vst_result": vst_result,
        "complete_result": complete_result,
        "raw_response": chat_result,
        "raw_content": raw_content,
        "answer": final_answer,
    }


vss_result = run_vss_pipeline(
    video_file=video_file,
    user_prompt=user_prompt,
    start_time=start_time,
    end_time=end_time,
)

print("VSS sensor_id:", vss_result["sensor_id"])
print("VSS answer:", vss_result["answer"])


VSS sensor_id: 37ccbd28-4c52-4cbf-b2c0-e2e8f7edda57
VSS answer: The video provides a detailed account of a formal parliamentary session. Here's an extensive summary:

**Setting and Environment**  
The session occurs in a structured chamber with wooden paneling and rows of desks equipped with microphones and documents. The atmosphere is professional and serious, with participants dressed formally, underscoring the significance of the proceedings.

**Key Participants**  
- A man in a suit delivers a speech from a central podium, using gestures to emphasize points.  
- Assembly members are seated at desks, engaged in activities such as taking notes, listening attentively, or participating in quiet discussions.  

**Visual Elements**  
- A large digital display board prominently shows voting results in multiple languages:  
  - Categories: "Ja / Oui / Sì" (Yes), "Nein / Non / No" (No), and "Enth. / Abst. / Ast." (Abstentions).  
  - The board dynamically updates vote tallies and includes a

# ASR Pipeline

In [22]:
# -------------------------------------------------------------------
# Run ASR
# -------------------------------------------------------------------

extract_pcm_from_video(
    video_file=video_file,
    pcm_file=PCM_OUTPUT_PATH,
    start_time=start_time,
    end_time=end_time,
 )

asr_result = run_riva_asr(PCM_OUTPUT_PATH)

riva_result = asr_result["raw_response"]
asr_segments = asr_result["segments"]
asr_transcript = asr_result["transcript"]

print("ASR diarization:")
for segment in asr_segments:
    print(
        f"[{segment['start_time']} - {segment['end_time']}] "
        f"Speaker {segment['speaker'] if segment['speaker'] is not None else '?'}: "
        f"{segment['text']}"
    )

print("\nASR transcript:")
print(asr_transcript)


ASR diarization:
[51680 - 55760] Speaker ?: Geschätzte Kolleginnen und Kollegen begrüße Sie zu keine Kommissar.
[70320 - 71040] Speaker 1: Herr Bischof.
[88080 - 88240] Speaker 1: Frau
[112080 - 119600] Speaker 1: Muller ragazzi road salzman.
[120240 - 127680] Speaker 1: Ist entschuldigt die Herren schmied schwanderfallen.
[136160 - 143520] Speaker ?: Schlussabstimmungen. Wir haben zu dreiundzwanzig Entwürfen eine Schlussabstimmung durchzuführen. Wer dem Entwurf zustimmt.
[148720 - 151520] Speaker ?: Der Präsident stimmt nicht mit Enthaltungen.
[152160 - 157440] Speaker ?: Werden mitgezählt zwanzig.
[160160 - 167520] Speaker ?: Jetzt über die obligatorische Arbeitslosenversicherung und die Insolvententschädigung Arbeitslosenversicherung für Arbeitnehmen.
[168240 - 175520] Speaker ?: in Arbeitgeber Stellung Ja, bedeutet Annahme, Nein, Ablehnung, Sie können stimmen.
[190400 - 191840] Speaker ?: Sie haben den Bundeslos.
[206720 - 207440] Speaker ?: In der französischen
[208480 - 215440] S

# Final Outputs

In [23]:
def get_nemotron_model_id():
    models_url = f"{NEMOTRON_BASE_URL}/models"
    try:
        response = requests.get(models_url, timeout=REQUEST_TIMEOUT)
        response.raise_for_status()
        payload = response.json()
        data = payload.get("data", []) if isinstance(payload, dict) else []

        model_ids = []
        for item in data:
            if isinstance(item, dict) and item.get("id"):
                model_ids.append(item["id"])

        for preferred in (NEMOTRON_DEFAULT_MODEL, "nvidia/nvidia-nemotron-nano-9b-v2"):
            if preferred in model_ids:
                return preferred

        if model_ids:
            return model_ids[0]
    except Exception:
        pass

    return NEMOTRON_DEFAULT_MODEL


def run_nemotron_synthesis(question, vss_answer, asr_transcript, asr_segments, sensor_id):
    model_id = get_nemotron_model_id()
    messages = build_nemotron_messages(question, vss_answer, asr_transcript, asr_segments)

    response = requests.post(
        f"{NEMOTRON_BASE_URL}/chat/completions",
        json={
            "model": model_id,
            "messages": messages,
            "temperature": 0.0,
            "max_tokens": 512,
            "chat_template_kwargs": {
                "enable_thinking": False,
            },
        },
        timeout=REQUEST_TIMEOUT,
    )
    response.raise_for_status()

    payload = response.json()
    choices = payload.get("choices", [])
    if not choices:
        raise RuntimeError(f"Nemotron returned no choices: {payload}")

    message = choices[0].get("message", {})
    final_answer = message.get("content", "").strip()
    if not final_answer:
        raise RuntimeError(f"Nemotron returned an empty message: {payload}")

    return {
        "model_id": model_id,
        "raw_response": payload,
        "answer": final_answer,
        "sensor_id": sensor_id,
    }


nemotron_result = run_nemotron_synthesis(
    question=user_prompt,
    vss_answer=vss_result["answer"],
    asr_transcript=asr_transcript,
    asr_segments=asr_segments,
    sensor_id=vss_result["sensor_id"],
)

final_answer = nemotron_result["answer"]

final_context = {
    "video_path": str(video_file),
    "time_range": {
        "start_time": start_time,
        "end_time": end_time,
    },
    "vss": {
        "sensor_id": vss_result["sensor_id"],
        "answer": vss_result["answer"],
        "raw_response": vss_result["raw_response"],
    },
    "asr": {
        "pcm_file": str(PCM_OUTPUT_PATH),
        "transcript": asr_transcript,
        "segments": asr_segments,
        "raw_response": riva_result,
    },
    "nemotron": {
        "model_id": nemotron_result["model_id"],
        "answer": final_answer,
        "raw_response": nemotron_result["raw_response"],
    },
}

display(Markdown("## Separate Outputs"))
display(Markdown(f"**VSS answer:** {vss_result['answer']}"))
display(Markdown(f"**ASR transcript:** {asr_transcript}"))
display(Markdown(f"**Nemotron final answer:** {final_answer}"))

print(json.dumps({
    "vss_answer": vss_result["answer"],
    "asr_transcript": asr_transcript,
    "nemotron_final_answer": final_answer,
    "nemotron_model_id": nemotron_result["model_id"],
}, indent=2, ensure_ascii=False))


## Separate Outputs

**VSS answer:** The video provides a detailed account of a formal parliamentary session. Here's an extensive summary:

**Setting and Environment**  
The session occurs in a structured chamber with wooden paneling and rows of desks equipped with microphones and documents. The atmosphere is professional and serious, with participants dressed formally, underscoring the significance of the proceedings.

**Key Participants**  
- A man in a suit delivers a speech from a central podium, using gestures to emphasize points.  
- Assembly members are seated at desks, engaged in activities such as taking notes, listening attentively, or participating in quiet discussions.  

**Visual Elements**  
- A large digital display board prominently shows voting results in multiple languages:  
  - Categories: "Ja / Oui / Sì" (Yes), "Nein / Non / No" (No), and "Enth. / Abst. / Ast." (Abstentions).  
  - The board dynamically updates vote tallies and includes a visual seating arrangement with colored dots indicating individual voting statuses.  

**Proceedings**  
- The session begins with the speaker addressing the assembly.  
- Camera cuts frequently between the speaker and assembly members, capturing interactions and focus shifts.  
- The voting process is central to the video, with the display board reflecting real-time changes in vote counts.  
- The session concludes with the speaker continuing his address, maintaining a composed demeanor.  

**Overall Tone**  
The proceedings are serious and methodical, suggesting a critical legislative or governmental decision-making process. The use of multilingual voting options and detailed visual aids highlights the formal and inclusive nature of the event.

**ASR transcript:** Geschätzte Kolleginnen und Kollegen begrüße Sie zu keine Kommissar.
Herr Bischof.
Frau
Muller ragazzi road salzman.
Ist entschuldigt die Herren schmied schwanderfallen.
Schlussabstimmungen. Wir haben zu dreiundzwanzig Entwürfen eine Schlussabstimmung durchzuführen. Wer dem Entwurf zustimmt.
Der Präsident stimmt nicht mit Enthaltungen.
Werden mitgezählt zwanzig.
Jetzt über die obligatorische Arbeitslosenversicherung und die Insolvententschädigung Arbeitslosenversicherung für Arbeitnehmen.
in Arbeitgeber Stellung Ja, bedeutet Annahme, Nein, Ablehnung, Sie können stimmen.
Sie haben den Bundeslos.
In der französischen
Resistance und italienischen Widerstand Annahme, nein bedeutet ab
Schuldigung war mein Fehler
Sie haben den Bundesgesetz zugestimmt mit vierzig gegen zwei Stimmen bei zwei Enthaltungen zweiundzwanzig.
Bundesgesetz über die Landwirtschaft vereinfachte Zulassung von Pflanzenschutzmitteln.
Sie können stimmen.
Sie haben den Bundesgesetz zugestimmt mit zweiunddreißig gegen zwölf Stimmen bei Null Enthaltungen vierundzwanzig, Null fünfundsechzig
Bundesgesetz über Schuldbetreibung und Konkurs, Betreibungsauskunft, elektronische Zustellung und Online Versteigerung.
Sie können stimmen.
Sie haben den Bundesgesetz zugestimmt mit dreiundvierzig gegen eine Stimme bei
Enthaltungen Bundesbeschluss über die Zusatzfinanzierung der
Durch eine Erhöhung der Mehrwertsteuer. Ja, bedeutet Danahme Nein, Ablehnung, Sie können stimmen
Sie haben den Bundesbeschloss zugestimmt mit achtundzwanzig
Gegen dreizehn Stimmen bei drei Enthaltungen vierundzwanzig null achtzig Bundesbeschluss über die Volksinitiative
Sie haben den Bundesbeschluss mit vierzig gegen drei Stimmen bei einer Enthaltung zugestimmt. Vierundzwanzig null neunzig Strahlenschutz.
Gesetz ja bedeutet Annahme die Ablehnung.
Sie können stimmen.
Sie haben
dem Gesetz zugestimmt mit dreiundvierzig gegen eine Stimme bei null Enthaltungen vierundzwanzig
Unterkantonalen mindestlöhnen liegen. Sie können stimmen.
Sie haben mit dreiundzwanzig gegen
Sechzehn Stimmen bei fünf Enthaltungen diesem Bundesgesetz zugestimmt. Fünfundzwanzig null achtzehn Bundesbeschluss.
abschaffen. Sie können stimmen
Sie haben den Bundesbeschluss mit vierundzwanzig.
Gegen zwanzig Stimmen bei null Enthaltungen zugestimmt. Fünfundzwanzig null neunzehn Bundesgesetz über Schuld
Sie haben den Bundesgesetz zugestimmt mit vierunddreißig.
Enthaltungen fünfundzwanzig, Null siebenundvierzig Bundesgesetz über die politischen Rechte
Können jetzt stimmen
Sie haben dem Bundesgesetz zugestimmt mit siebenunddreißig gegenüber.
Sieben Stimmen bei null Enthaltungen fünfundzwanzig null achtundsechzig Entwurf eins.
Stoppen, sie können stimmen.
Sie haben mit achtundzwanzig gegen vierzehn Stimmen bei zwei Enthaltungen zugestimmt. Entwurf
Bundesbeschluß über die eigenössische Volksinitiative. Jederzeit Strom für alle Blackout stoppen
Sie können stimmen.
Sie haben den Bundesbeschloss zugestimmt mit zweiunddreißig gegen zehn Stimmen bei zwei Enthaltungen, fünfundzwanzig null einundsiebzig.
Bundesgesetz über die eigenössische Finanzmarktaufsicht.
Sie können stimmen.
Sie haben dem Gesetz zugestimmt mit achtunddreißig gegen fünf Stimmen bei einer Enthaltung.
Sie können stimmen
Fünfundzwanzig.
Bundesgesetz über die wirtschaftliche Landesversorgung
Jetzt können sie stimmen.
Sie haben dem Bundesgesetzung.
Zugestimmt mit vierundvierzig gegen Nullstimmen bei Null Enthaltungen.
Sie können stimmen
gegen eine Stimme bei einer Enthaltung fünfundzwanzig Bundesbeschluß.
Über die Genehmigung des umfassenden Abkommens über die Förderung und den Schutz
Stimmen
Über die Förderung und den Schutz von Investitionen zwischen der schweizerischen Eigenossenschaft und der Republik Chile.
Jetzt könnet sich stimmen.
Sie haben den Bundesbeschluß mit vierundvierzig.
Fünfundzwanzig vierhundertzweise.
Jetzt können sie stimmen.
Bei zwei Enthaltungen fünfundzwanzig Bundesgesetz über den Umweltschutz
Sie können stimmen.
Sie haben den Bundesgesetz zugestimmt mit vierundvierzig gegen Null Stimmen bei Null Enthaltungen sechsundzwanzig
Sie können stimmen.
Sie haben diesem Bundesbeschloss mit sechsunddreißig gegen drei Stimmen bei fünf Enthaltungen zu gestehen und zum
Sie können stimmen.
Sie haben auf diesem Bundesbeschluß zugestimmt und das mit einunddreißig gegen
Sowitz sind wir am Ende der Sommersaison.
Herzlich dafür zu danken, dass es uns gelungen ist, alle traktiert traktandierten Geschäfte dieser session auch zu
Und keine auf die nächste Session verschieben zu müssen. Geht davon aus, dass wir in der nächsten Saission im September dann ein doch.
stark befrachtetes Programm haben werden und bin deshalb froh, dass wir da nicht noch Geschäfte aus dieser Session da.
Und bevor Sie in die wohlverdienten Sommerferien gehen, bedenken Sie
Wenn Steine bedenken Sie, Steine rollen den Abhang hinunter, der Rauch steigt auf

**Nemotron final answer:** Okay, let's tackle this. The user wants an extensive and detailed summary of the video based on the VSS and ASR provided. First, I need to parse both the VSS and ASR to extract key information.

The VSS describes a formal parliamentary session with a structured setting, key participants like a speaker in a suit and assembly members, visual elements like a multilingual voting board, and the proceedings involving speeches, voting on multiple bills, and a serious tone. The ASR transcript is a mix of German and some other languages, with the speaker discussing various legislative decisions, voting results, and some personal remarks at the end.

I need to combine these. The summary should cover the setting, participants, visual aids, the main proceedings (voting on several bills), and the overall tone. The ASR gives specifics on the bills passed, like the ones on agriculture, unemployment insurance, environmental protection, etc., along with vote counts. The VSS mentions the multilingual voting options and the formal atmosphere.

I should structure the summary to include all these elements without missing any key points. Make sure to mention the multilingual voting (Ja/Oui/Sì, Nein/Non, etc.), the number of bills discussed (21 as per VSS), the specific bills passed with their vote counts from the ASR, and the conclusion where the speaker thanks everyone and mentions future sessions. Also, note the serious and methodical tone throughout.

Need to check if there are any discrepancies between VSS and ASR. For example, the VSS says 21 drafts, and the ASR lists several bills. The ASR also has some unclear parts, but the main points are the voting outcomes. Ensure that the summary is comprehensive, detailed, and flows well, combining both sources accurately.
</think>

The video documents a formal parliamentary session in a structured chamber with wooden paneling and rows of desks equipped with microphones. Participants, dressed formally, engage in a serious and methodical legislative process. A central speaker delivers speeches while assembly members take notes or discuss quietly. A large digital display board shows real-time multilingual voting results (Ja/Oui/Sì, Nein/Non, Enth./Abst.) with dynamic vote tallies and seating visualizations. The session involves voting on 21 legislative proposals, including bills on agriculture, unemployment insurance, environmental protection, and financial regulations. Key outcomes include unanimous or majority approvals for most bills, with specific vote counts (e.g

{
  "vss_answer": "The video provides a detailed account of a formal parliamentary session. Here's an extensive summary:\n\n**Setting and Environment**  \nThe session occurs in a structured chamber with wooden paneling and rows of desks equipped with microphones and documents. The atmosphere is professional and serious, with participants dressed formally, underscoring the significance of the proceedings.\n\n**Key Participants**  \n- A man in a suit delivers a speech from a central podium, using gestures to emphasize points.  \n- Assembly members are seated at desks, engaged in activities such as taking notes, listening attentively, or participating in quiet discussions.  \n\n**Visual Elements**  \n- A large digital display board prominently shows voting results in multiple languages:  \n  - Categories: \"Ja / Oui / Sì\" (Yes), \"Nein / Non / No\" (No), and \"Enth. / Abst. / Ast.\" (Abstentions).  \n  - The board dynamically updates vote tallies and includes a visual seating arrangement